# 01 — SDNET2018 baseline (YOLOv8n-cls) on Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hasanbahcecii/surface-anomaly-detector/blob/main/notebooks/01_train_baseline_colab.ipynb)

**Runtime → Change runtime type → T4 GPU** before running.

Before the first run, add two Colab secrets (🔑 icon in the left sidebar) with notebook access enabled:
- `KAGGLE_USERNAME`
- `KAGGLE_KEY` (from kaggle.com → Settings → API → Create New Token)

Data lives on the fast local disk (`/content`); weights and metrics are copied to Google Drive so they survive the session.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Setup

In [ ]:
REPO_URL = "https://github.com/hasanbahcecii/surface-anomaly-detector.git"
DRIVE_OUT = "/content/drive/MyDrive/surface-anomaly-detector"

import os
if not os.path.exists("/content/surface-anomaly-detector"):
    !git clone -q {REPO_URL} /content/surface-anomaly-detector
    if not os.path.exists("/content/surface-anomaly-detector"):
        raise RuntimeError("git clone failed - is the GitHub repo public?")
%cd /content/surface-anomaly-detector
!pip install -q -r requirements.txt

from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_OUT, exist_ok=True)

## 2. Download SDNET2018 from Kaggle

In [ ]:
from google.colab import userdata
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

KAGGLE_DATASET = "aniruddhsharma/structural-defects-network-concrete-crack-images"
if not os.path.exists("data/raw"):
    !kaggle datasets download -d {KAGGLE_DATASET} -p data/ --unzip -q
    # Kaggle unzips into a dataset-named folder; normalize to data/raw
    !find data -maxdepth 1 -mindepth 1 -type d ! -name raw ! -name processed -exec mv {} data/raw \;
!find data/raw -maxdepth 3 -type d | head -30

## 3. Train / val / test split (70 / 15 / 15, stratified by surface × label)

In [ ]:
!python scripts/prepare_data.py --src data/raw --out data/processed --seed 42
!cp data/processed/split_manifest.csv {DRIVE_OUT}/

## 4. Fine-tune YOLOv8n-cls

SDNET2018 is image-level labeled (no bounding boxes), so the baseline is a **patch classifier**.
In v0.3 it is run as a sliding window over large surfaces to produce a crack heatmap.

In [ ]:
from ultralytics import YOLO

# Absolute path on Drive: checkpoints survive a Colab disconnect, and newer
# Ultralytics versions don't nest it under runs_dir like a relative path.
RUN_DIR = f"{DRIVE_OUT}/runs"
model = YOLO("yolov8n-cls.pt")
results = model.train(
    data="data/processed",
    epochs=30,
    imgsz=224,
    batch=128,
    patience=8,
    seed=42,
    project=RUN_DIR,
    name="baseline",
    exist_ok=True,
)
SAVE_DIR = str(model.trainer.save_dir)
print("saved to", SAVE_DIR)

If Colab disconnects mid-training, re-run sections 1–3, skip the training cell above, and run the cell below with `RESUME = True` to continue from the last checkpoint.

In [ ]:
# Resume an interrupted run: set RESUME = True after a disconnect
RESUME = False
if RESUME:
    from ultralytics import YOLO
    SAVE_DIR = f"{DRIVE_OUT}/runs/baseline"
    model = YOLO(f"{SAVE_DIR}/weights/last.pt")
    results = model.train(resume=True)

## 5. Evaluate on the held-out test split

In [ ]:
!python scripts/evaluate.py --weights {SAVE_DIR}/weights/best.pt --data data/processed --split test --out {SAVE_DIR}/test_metrics.json

## 6. Check outputs on Drive

In [ ]:
!ls -la {SAVE_DIR} {SAVE_DIR}/weights